<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>R Analysis</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook compares per-sample small-variant QC across PEPPER-Margin-DeepVariant, PAV, and Clair3 on T2T. It plots cumulative Ti/Tv versus QUAL, SNP counts, and SNP-count versus Ti/Tv, highlighting caller-specific QUAL thresholds.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
install.packages("janitor")

In [ ]:
library(reticulate)
library(bigrquery)
library(ggplot2)
library(tidyverse)
library(ggpubr)
library(vcfR)
library(janitor)

In [ ]:
# The FISS (FireCloud) Python API, called from R through reticulate, for reading Terra data tables.
fapi = import('firecloud.api')

In [ ]:
# Workspace details Terra sets as environment variables.
BILLING_PROJECT_ID = Sys.getenv('GOOGLE_PROJECT')       # the Google billing project
WORKSPACE_BUCKET = Sys.getenv('WORKSPACE_BUCKET')       # the workspace bucket
WORKSPACE_NAME = Sys.getenv('WORKSPACE_NAME')           # workspace name
WORKSPACE_NAMESPACE = Sys.getenv('WORKSPACE_NAMESPACE') # workspace namespace

In [ ]:
# Load a Terra data table into a tibble, converting 'true'/'false' list columns to logicals.
load_workspace_table <- function(table_name) {
    a = fapi$get_entities(WORKSPACE_NAMESPACE, WORKSPACE_NAME, table_name)

    json = a$json()
    df = tibble(metadata = json) %>% unnest_wider(metadata) %>% unnest_wider(attributes)

    for (name in df %>% select(where(is.list)) %>% names()) {
        q = as.vector(unlist(df[,name])) %in% c("true", "TRUE", "false", "FALSE") %>% unique()
        if (q) {
            df[,name] = as.logical(as.vector(unlist(df[,name])))
        }
    }
    
    return(df)
}

In [ ]:
# Per-sample table for the T2T-aligned samples; the *_small_variant_stats columns point to each caller's
# bcftools-stats-style summary.
ha_sample_t2t_df = load_workspace_table('ha-sample-t2t')

In [ ]:
ha_sample_t2t_df

In [ ]:
# PEPPER-Margin-DeepVariant (dvp) stats for one sample: keep the QUAL section (SNP, Ti, Tv, and indel
# counts per quality value) and compute Ti/Tv.
system(paste("gsutil cp ", ha_sample_t2t_df$dvp_small_variant_stats[1], " .", sep=""), intern=TRUE)
system(paste("grep 'QUAL\t' ", basename(ha_sample_t2t_df$dvp_small_variant_stats[1]), " | sed 's/ /_/g' > test.txt"), intern=TRUE)

dvp_df = read_table("test.txt")
names(dvp_df) = c("QUAL", "id", "quality", "num_snps", "num_ti", "num_tv", "num_indels")
dvp_df = dvp_df %>% mutate(ti_tv = num_ti/num_tv)
dvp_df

In [ ]:
# Cumulative Ti/Tv of SNVs at or above each QUAL threshold, marking the QUAL 40 cutoff and Ti/Tv 2.1.
plot(0, 0, xlim=c(0, 100), ylim=c(0, 3), xlab="QUAL (Cumulative)", ylab="Ti/Tv", las=1, cex=1.3, cex.axis=1.3, cex.lab=1.3, bty="n")

for (i in seq(0, 100, by=1)) {
    dvp_sub_df = dvp_df %>% filter(quality >= i) %>% adorn_totals("row") %>% mutate(ti_tv = num_ti/num_tv) %>% tail(n=1)
    points(i, dvp_sub_df$ti_tv)
    
    if (i == 40) {
        points(40, dvp_sub_df$ti_tv, col="red", pch=19)
        text(40, 2.0, paste("num_snps =", dvp_sub_df$num_snps), pos=4)
    }
}

abline(h=2.1, col="red", lty=2)
abline(v=40, col="darkgreen", lty=2)

In [ ]:
# Download every sample's DeepVariant stats and extract the QUAL section.
for (file in (ha_sample_t2t_df %>% filter(!is.na(dvp_small_variant_stats)) %>% pull(dvp_small_variant_stats))) {
    if (!file.exists(basename(file))) {
        print(file)
        system(paste("gsutil cp ", file, " .", sep=""), intern=TRUE)
        system(paste("grep 'QUAL\t' ", basename(file), " | sed 's/ /_/g' > ", basename(file), ".qual.txt", sep=""), intern=TRUE)
    }
}

In [ ]:
system("ls | grep 'qual.txt'", intern=TRUE)

In [ ]:
# For each sample, totals over QUAL >= 40: SNV count and Ti/Tv.
dvp_qual40_df = NULL
for (file in (ha_sample_t2t_df %>% filter(!is.na(dvp_small_variant_stats)) %>% pull(dvp_small_variant_stats))) {
    qual_file = paste(basename(file), ".qual.txt", sep="")
    if (file.exists(basename(qual_file))) {
        dvp_df = read_table(qual_file, show_col_types=FALSE)
        names(dvp_df) = c("QUAL", "id", "quality", "num_snps", "num_ti", "num_tv", "num_indels")
        
        row_df = dvp_df %>% filter(quality >= 40) %>% adorn_totals("row") %>% mutate(ti_tv = num_ti/num_tv) %>% tail(n=1)
        if (is.null(row_df)) {
            dvp_qual40_df = row_df
        } else {
            dvp_qual40_df = rbind(dvp_qual40_df, row_df)
        }
    }
}

In [ ]:
# SNV counts (sorted) and Ti/Tv vs SNV count across samples.
barplot(sort(dvp_qual40_df$num_snps), border=NA)

In [ ]:
par(mar=c(5, 7, 4, 2))
plot(dvp_qual40_df$num_snps, dvp_qual40_df$ti_tv, las=1, cex.lab=1.3, cex.axis=1.3, xlab="Num SNVs (QUAL >= 40)", ylab="", bty="n")
mtext("Ti/Tv", 2, line=4, cex=1.3)


In [ ]:
# Same for PAV (assembly-based calls), which have no QUAL filter.
for (file in (ha_sample_t2t_df %>% filter(!is.na(pav_small_variant_stats)) %>% pull(pav_small_variant_stats))) {
    if (!file.exists(basename(file))) {
        system(paste("gsutil cp ", file, " .", sep=""), intern=TRUE)
        system(paste("grep 'QUAL\t' ", basename(file), " | sed 's/ /_/g' > ", basename(file), ".qual.txt", sep=""), intern=TRUE)
    }
}

In [ ]:
system("cat pav_example_sample.stats.txt.qual.txt", intern=TRUE)

In [ ]:
# Scratch cell: pav_df is not defined in this copy.
paste(basename((ha_sample_t2t_df %>% filter(!is.na(pav_small_variant_stats)) %>% pull(pav_small_variant_stats))[1]), ".qual.txt", sep="")

pav_df


In [ ]:
# Per-sample PAV SNV counts and Ti/Tv.
pav_all_df = NULL
for (file in (ha_sample_t2t_df %>% filter(!is.na(pav_small_variant_stats)) %>% pull(pav_small_variant_stats))) {
    qual_file = paste(basename(file), ".qual.txt", sep="")
    if (file.exists(basename(qual_file))) {
        row_df = read_table(qual_file, show_col_types=FALSE)
        names(row_df) = c("QUAL", "id", "quality", "num_snps", "num_ti", "num_tv", "num_indels")
        row_df = row_df %>% mutate(ti_tv = num_ti/num_tv)
        if (is.null(row_df)) {
            pav_all_df = row_df
        } else {
            pav_all_df = rbind(pav_all_df, row_df)
        }
    }
}

In [ ]:
par(mar=c(5, 7, 4, 2))
plot(pav_all_df$num_snps, pav_all_df$ti_tv, las=1, cex.lab=1.3, cex.axis=1.3, xlab="Num SNVs", ylab="", bty="n")
mtext("Ti/Tv", 2, line=4, cex=1.3)

In [ ]:
barplot(sort(pav_all_df$num_snps), border=NA)

In [ ]:
# DeepVariant and PAV side by side on the same axes.
par(mar=c(5, 7, 4, 2))
plot(dvp_qual40_df$num_snps, dvp_qual40_df$ti_tv, las=1, cex.lab=1.3, cex.axis=1.3, xlab="Num SNVs (QUAL >= 40)", ylab="", bty="n", xlim=c(0, 4e6), ylim=c(1.5, 2.15), main="PEPPER-Margin-DeepVariant")
mtext("Ti/Tv", 2, line=4, cex=1.3)

par(mar=c(5, 7, 4, 2))
plot(pav_all_df$num_snps, pav_all_df$ti_tv, las=1, cex.lab=1.3, cex.axis=1.3, xlab="Num SNVs", ylab="", bty="n", xlim=c(0, 4e6), ylim=c(1.5, 2.15), main="PAV")
mtext("Ti/Tv", 2, line=4, cex=1.3)

In [ ]:
# Same for Clair3.
for (file in (ha_sample_t2t_df %>% filter(!is.na(clair_small_variant_stats)) %>% pull(clair_small_variant_stats))) {
    if (!file.exists(basename(file))) {
        print(file)
        system(paste("gsutil cp ", file, " .", sep=""), intern=TRUE)
        system(paste("grep 'QUAL\t' ", basename(file), " | sed 's/ /_/g' > ", basename(file), ".qual.txt", sep=""), intern=TRUE)
    }
}

In [ ]:
# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA <- Sys.getenv("AOU_LR_DATA")
if (AOU_LR_DATA == "") stop("Set the AOU_LR_DATA environment variable before running this notebook")

In [ ]:
# One sample's Clair3 stats. The next cell reads the .qual.txt file made by the download loop above.
system(paste0("gsutil cat ", AOU_LR_DATA, "/outputs/T2T/SummarizeSmallVariants/example_sample/example_sample.clair.stats.txt"), intern=TRUE)

In [ ]:
clair_df = read_table("example_sample.clair.stats.txt.qual.txt")
names(clair_df) = c("QUAL", "id", "quality", "num_snps", "num_ti", "num_tv", "num_indels")
clair_df = clair_df %>% mutate(ti_tv = num_ti/num_tv)
clair_df

In [ ]:
# Clair3 uses a QUAL >= 26 cutoff; cumulative Ti/Tv by threshold for this sample.
clair_df %>% filter(quality >= 26) %>% adorn_totals("row") %>% tail(n=1)

In [ ]:
plot(0, 0, xlim=c(0, 100), ylim=c(0, 3), xlab="QUAL (Cumulative)", ylab="Ti/Tv", las=1, cex=1.3, cex.axis=1.3, cex.lab=1.3, bty="n")

for (i in seq(0, 100, by=1)) {
    clair_sub_df = clair_df %>% filter(quality >= i) %>% adorn_totals("row") %>% mutate(ti_tv = num_ti/num_tv) %>% tail(n=1)
    points(i, clair_sub_df$ti_tv)
    
    if (i == 26) {
        points(26, clair_sub_df$ti_tv, col="red", pch=19)
        text(26, 2.0, paste("num_snps =", clair_sub_df$num_snps), pos=4)
    }
}

abline(h=2.1, col="red", lty=2)
abline(v=26, col="darkgreen", lty=2)

In [ ]:
# For each sample, Clair3 totals over QUAL >= 26: SNV count and Ti/Tv.
clair_qual26_df = NULL
for (file in (ha_sample_t2t_df %>% filter(!is.na(clair_small_variant_stats)) %>% pull(clair_small_variant_stats))) {
    qual_file = paste(basename(file), ".qual.txt", sep="")
    if (file.exists(basename(qual_file))) {
        clair_df = read_table(qual_file, show_col_types=FALSE)
        names(clair_df) = c("QUAL", "id", "quality", "num_snps", "num_ti", "num_tv", "num_indels")
        
        row_df = clair_df %>% filter(quality >= 26) %>% adorn_totals("row") %>% mutate(ti_tv = num_ti/num_tv) %>% tail(n=1)
        if (is.null(row_df)) {
            clair_qual26_df = row_df
        } else {
            clair_qual26_df = rbind(clair_qual26_df, row_df)
        }
    }
}

In [ ]:
# Clair3 SNV counts and Ti/Tv (the x-axis label says QUAL >= 40, but the cutoff is 26).
barplot(sort(clair_qual26_df$num_snps), border=NA)

In [ ]:
par(mar=c(5, 7, 4, 2))
plot(clair_qual26_df$num_snps, clair_qual26_df$ti_tv, las=1, cex.lab=1.3, cex.axis=1.3, xlab="Num SNVs (QUAL >= 40)", ylab="", bty="n")
mtext("Ti/Tv", 2, line=4, cex=1.3)

In [ ]:
# The three callers side by side on the same axes.
par(mar=c(5, 7, 4, 2))
plot(dvp_qual40_df$num_snps, dvp_qual40_df$ti_tv, las=1, cex.lab=1.3, cex.axis=1.3, xlab="Num SNVs (QUAL >= 40)", ylab="", bty="n", xlim=c(0, 4e6), ylim=c(1.5, 2.15), main="PEPPER-Margin-DeepVariant")
mtext("Ti/Tv", 2, line=4, cex=1.3)

par(mar=c(5, 7, 4, 2))
plot(pav_all_df$num_snps, pav_all_df$ti_tv, las=1, cex.lab=1.3, cex.axis=1.3, xlab="Num SNVs", ylab="", bty="n", xlim=c(0, 4e6), ylim=c(1.5, 2.15), main="PAV")
mtext("Ti/Tv", 2, line=4, cex=1.3)

par(mar=c(5, 7, 4, 2))
plot(clair_qual26_df$num_snps, clair_qual26_df$ti_tv, las=1, cex.lab=1.3, cex.axis=1.3, xlab="Num SNVs (QUAL > 26)", ylab="", bty="n", xlim=c(0, 4e6), ylim=c(1.5, 2.15), main="Clair3")
mtext("Ti/Tv", 2, line=4, cex=1.3)